In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
!pip install /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
!pip install /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl

Processing /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
autograd is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Processing /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
  Preparing metadata (setup.py) ... done
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4031 sha256=3aff0bbdd62d6dbae3c7eceb87a279600e8d826e2e9e8fc7e2a1e78a7306fb26
  Stored in directory: /root/.cache/pip/wheels/6b/b5/e0/4c79e15c0b5f2c15ecf613c720bb20daab20a666eb67135155
Successfully built autograd-gamma
Processing /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl


In [3]:
train = pd.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv')
test = pd.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')
sub = pd.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/sample_submission.csv')

In [4]:
train.dropna(inplace = True)
train = train.sample(frac=0.5, random_state=42)  # 訓練データを50%に縮小

train.head()

,ID,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,...,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs,efs_time
26652,26652,High,No,Poor,No,2.0,8.0,No TBI,No,6.0,...,Permissive mismatched,Unrelated,"N/A, Mel not given",8.0,No,2.0,Yes,10.0,1.0,6.985
25633,25633,High - TED AML case <missing cytogenetics,No,Intermediate,No,2.0,8.0,No TBI,No,6.0,...,GvH non-permissive,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,22.647
9946,9946,High,No,Poor,Yes,2.0,8.0,No TBI,No,6.0,...,Permissive mismatched,Multiple donor (non-UCB),"N/A, Mel not given",8.0,No,2.0,No,10.0,1.0,3.320
5225,5225,Intermediate,No,Poor,No,2.0,8.0,No TBI,No,6.0,...,Permissive mismatched,Unrelated,MEL,8.0,No,2.0,No,10.0,0.0,27.887
25365,25365,High,No,Poor,No,2.0,8.0,No TBI,No,6.0,...,GvH non-permissive,Related,MEL,8.0,No,2.0,Yes,10.0,1.0,5.966


In [5]:
# two target in the train dataset so we havr to convrt it one
# !pip install lifelines
from lifelines import KaplanMeierFitter

def transform_survival_probability(df, time_col='efs_time', event_col='efs'):

    kmf = KaplanMeierFitter()

    kmf.fit(df[time_col], event_observed=df[event_col])

    survival_probabilities = kmf.survival_function_at_times(df[time_col]).values.flatten()

    censored_mask = df[event_col] == 0

    return survival_probabilities

train["target"] = transform_survival_probability(train, time_col='efs_time', event_col='efs')

drop_cols = ["ID", 'efs', 'efs_time']
train = train.drop(columns=[col for col in drop_cols if col in train.columns])
test = test.drop(columns=[col for col in drop_cols if col in test.columns])

train.head()

,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,...,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,target
26652,High,No,Poor,No,2.0,8.0,No TBI,No,6.0,Peripheral blood,...,No,Permissive mismatched,Unrelated,"N/A, Mel not given",8.0,No,2.0,Yes,10.0,0.577508
25633,High - TED AML case <missing cytogenetics,No,Intermediate,No,2.0,8.0,No TBI,No,6.0,Peripheral blood,...,No,GvH non-permissive,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.361206
9946,High,No,Poor,Yes,2.0,8.0,No TBI,No,6.0,Peripheral blood,...,No,Permissive mismatched,Multiple donor (non-UCB),"N/A, Mel not given",8.0,No,2.0,No,10.0,0.956434
5225,Intermediate,No,Poor,No,2.0,8.0,No TBI,No,6.0,Peripheral blood,...,No,Permissive mismatched,Unrelated,MEL,8.0,No,2.0,No,10.0,0.360034
25365,High,No,Poor,No,2.0,8.0,No TBI,No,6.0,Peripheral blood,...,No,GvH non-permissive,Related,MEL,8.0,No,2.0,Yes,10.0,0.729483


In [6]:
train.head()

,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,...,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,target
26652,High,No,Poor,No,2.0,8.0,No TBI,No,6.0,Peripheral blood,...,No,Permissive mismatched,Unrelated,"N/A, Mel not given",8.0,No,2.0,Yes,10.0,0.577508
25633,High - TED AML case <missing cytogenetics,No,Intermediate,No,2.0,8.0,No TBI,No,6.0,Peripheral blood,...,No,GvH non-permissive,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.361206
9946,High,No,Poor,Yes,2.0,8.0,No TBI,No,6.0,Peripheral blood,...,No,Permissive mismatched,Multiple donor (non-UCB),"N/A, Mel not given",8.0,No,2.0,No,10.0,0.956434
5225,Intermediate,No,Poor,No,2.0,8.0,No TBI,No,6.0,Peripheral blood,...,No,Permissive mismatched,Unrelated,MEL,8.0,No,2.0,No,10.0,0.360034
25365,High,No,Poor,No,2.0,8.0,No TBI,No,6.0,Peripheral blood,...,No,GvH non-permissive,Related,MEL,8.0,No,2.0,Yes,10.0,0.729483


In [7]:
test.head()

,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,...,karnofsky_score,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10
0,N/A - non-malignant indication,No,NaN,No,NaN,NaN,No TBI,No,6.0,Bone marrow,...,90.0,No,NaN,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0
1,Intermediate,No,Intermediate,No,2.0,8.0,"TBI +- Other, >cGy",No,6.0,Peripheral blood,...,90.0,No,Permissive mismatched,Related,"N/A, Mel not given",8.0,No,2.0,Yes,10.0
2,N/A - non-malignant indication,No,NaN,No,2.0,8.0,No TBI,No,6.0,Bone marrow,...,90.0,No,Permissive mismatched,Related,"N/A, Mel not given",8.0,No,2.0,No,10.0


In [8]:
target = train['target']

train_numerical = train.select_dtypes(include=[np.number]).drop('target', axis = 1)
train_categorical = train.select_dtypes(exclude=[np.number])

In [9]:
target

26652    0.577508
25633    0.361206
9946     0.956434
5225     0.360034
25365    0.729483
           ...   
3330     0.440529
26248    0.911854
7894     0.897670
3396     0.851064
766      0.532898
Name: target, Length: 987, dtype: float64

In [10]:
train_numerical.head()

,hla_match_c_high,hla_high_res_8,hla_low_res_6,hla_high_res_6,hla_high_res_10,hla_match_dqb1_high,hla_nmdp_6,hla_match_c_low,hla_match_drb1_low,hla_match_dqb1_low,...,donor_age,hla_match_b_low,age_at_hct,hla_match_a_low,hla_match_b_high,comorbidity_score,karnofsky_score,hla_low_res_8,hla_match_drb1_high,hla_low_res_10
26652,2.0,8.0,6.0,6.0,10.0,2.0,6.0,2.0,2.0,2.0,...,39.093,2.0,33.860,2.0,2.0,2.0,70.0,8.0,2.0,10.0
25633,2.0,8.0,6.0,6.0,10.0,2.0,6.0,2.0,2.0,2.0,...,28.961,2.0,29.266,2.0,2.0,1.0,90.0,8.0,2.0,10.0
9946,2.0,8.0,6.0,6.0,10.0,2.0,6.0,2.0,2.0,2.0,...,31.955,2.0,32.205,2.0,2.0,1.0,70.0,8.0,2.0,10.0
5225,2.0,8.0,6.0,6.0,10.0,2.0,6.0,2.0,2.0,2.0,...,40.116,2.0,36.498,2.0,2.0,0.0,90.0,8.0,2.0,10.0
25365,2.0,8.0,6.0,6.0,10.0,2.0,6.0,2.0,2.0,2.0,...,53.094,2.0,64.308,2.0,2.0,2.0,70.0,8.0,2.0,10.0


In [11]:
train_numerical.fillna(train_numerical.mode(), inplace = True)

In [12]:
train_numerical.isnull().sum()

hla_match_c_high       0
hla_high_res_8         0
hla_low_res_6          0
hla_high_res_6         0
hla_high_res_10        0
hla_match_dqb1_high    0
hla_nmdp_6             0
hla_match_c_low        0
hla_match_drb1_low     0
hla_match_dqb1_low     0
year_hct               0
hla_match_a_high       0
donor_age              0
hla_match_b_low        0
age_at_hct             0
hla_match_a_low        0
hla_match_b_high       0
comorbidity_score      0
karnofsky_score        0
hla_low_res_8          0
hla_match_drb1_high    0
hla_low_res_10         0
dtype: int64

In [13]:
test_numerical = test.select_dtypes(include=[np.number])
test_categorical = test.select_dtypes(exclude=[np.number])

In [14]:
test_numerical.fillna(train_numerical.mode(), inplace = True)

In [15]:
for col in train_numerical.columns:
    print(train_numerical[col].value_counts())

hla_match_c_high
2.0    938
1.0     48
0.0      1
Name: count, dtype: int64
hla_high_res_8
8.0    758
7.0    123
6.0     43
5.0     36
4.0     27
Name: count, dtype: int64
hla_low_res_6
6.0    794
5.0    115
3.0     42
4.0     35
2.0      1
Name: count, dtype: int64
hla_high_res_6
6.0    760
5.0    128
4.0     51
3.0     47
2.0      1
Name: count, dtype: int64
hla_high_res_10
10.0    734
9.0     126
8.0      50
6.0      30
7.0      29
5.0      18
Name: count, dtype: int64
hla_match_dqb1_high
2.0    894
1.0     88
0.0      5
Name: count, dtype: int64
hla_nmdp_6
6.0    755
5.0    181
3.0     43
4.0      8
Name: count, dtype: int64
hla_match_c_low
2.0    926
1.0     61
Name: count, dtype: int64
hla_match_drb1_low
2.0    925
1.0     62
Name: count, dtype: int64
hla_match_dqb1_low
2.0    923
1.0     60
0.0      4
Name: count, dtype: int64
year_hct
2017    221
2016    179
2018    177
2015    130
2008     93
2013     63
2012     51
2014     25
2011     20
2010     14
2009      8
2019      6
N

いくつかの数値カラムは、ラベリングできるので、カテゴリカルとみなし、整数化してからカテゴリー型にする

In [16]:
# train_categorical.fillna(tr, inplace = True)
test_categorical.fillna(train_categorical.mode(), inplace = True)

In [17]:
all_data_categocical = pd.concat([train_categorical, test_categorical])

from sklearn.preprocessing import LabelEncoder


for col in train_categorical.columns:
    le = LabelEncoder()
    
    all_data_categocical[col] = le.fit_transform(all_data_categocical[col].astype('category'))

train_categorical = all_data_categocical[:len(train)]
test_categorical = all_data_categocical[len(train):]

In [18]:
# train_numericalの中には、離散値をとる場合がある。それを整数型にして、カテゴリ型に変換する
col_to_drop = ['year_hct','donor_age','age_at_hct']

train_numerical_categorical = train_numerical.drop(col_to_drop, axis = 1).astype(int).astype('category')
test_numerical_categorical = test_numerical.drop(col_to_drop, axis = 1).astype(int).astype('category')

train_numeric = train_numerical[col_to_drop]
test_numeric = test_numerical[col_to_drop]

train_category = pd.concat([train_numerical_categorical, train_categorical], axis = 1)
test_category = pd.concat([test_numerical_categorical, test_categorical], axis = 1)


In [19]:
train_df = pd.concat([train_category, train_numeric], axis = 1)
test_df = pd.concat([test_category, test_numeric], axis = 1)
train_df['target'] = target
test_df.isnull().sum()

hla_match_c_high          0
hla_high_res_8            0
hla_low_res_6             0
hla_high_res_6            0
hla_high_res_10           0
hla_match_dqb1_high       0
hla_nmdp_6                0
hla_match_c_low           0
hla_match_drb1_low        0
hla_match_dqb1_low        0
hla_match_a_high          0
hla_match_b_low           0
hla_match_a_low           0
hla_match_b_high          0
comorbidity_score         0
karnofsky_score           0
hla_low_res_8             0
hla_match_drb1_high       0
hla_low_res_10            0
dri_score                 0
psych_disturb             0
cyto_score                0
diabetes                  0
tbi_status                0
arrhythmia                0
graft_type                0
vent_hist                 0
renal_issue               0
pulm_severe               0
prim_disease_hct          0
cmv_status                0
tce_imm_match             0
rituximab                 0
prod_type                 0
cyto_score_detail         0
conditioning_intensi

In [20]:
import catboost as cb
from catboost import Pool, CatBoostRegressor

from sklearn.model_selection import train_test_split

In [21]:
X_train,X_test, y_train, y_test = train_test_split(train_df.drop('target', axis=1), train_df['target'], test_size=0.2, random_state=42, shuffle=True)

In [22]:
train_pool = Pool(data=X_train, label=y_train, cat_features=list(train_category.columns))
test_pool = Pool(data=X_test, label=y_test, cat_features=list(train_category.columns))

In [23]:
cbg = CatBoostRegressor(iterations=1000,
                        learning_rate = 0.05,
                        depth=6,
                        loss_function='RMSE',
                        eval_metric='RMSE',
                        random_seed=42,
                        od_type='Iter',
                        od_wait=50)

cbg.fit(train_pool, eval_set=test_pool, verbose=100, plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 0.2110907	test: 0.2171292	best: 0.2171292 (0)	total: 77.5ms	remaining: 1m 17s
100:	learn: 0.1799413	test: 0.2001792	best: 0.2001229 (97)	total: 1.41s	remaining: 12.5s
200:	learn: 0.1613925	test: 0.1955302	best: 0.1954373 (194)	total: 2.75s	remaining: 10.9s
300:	learn: 0.1472047	test: 0.1942018	best: 0.1937244 (271)	total: 4.17s	remaining: 9.68s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.1937243695
bestIteration = 271

Shrink model to first 272 iterations.


In [24]:
from sklearn.metrics import mean_squared_error
y_pred = cbg.predict(test_df)

np.sqrt(mean_squared_error(y_test, cbg.predict(X_test)))

0.1937243701276626

In [25]:
catboost_pred = cbg.predict(test_df)
catboost_pred

array([0.45874413, 0.54450583, 0.5243763 ])

In [26]:
sub['prediction'] = catboost_pred
# sub.to_csv('submission_catboost.csv', index=False)

In [27]:
sub

,ID,prediction
0,28800,0.458744
1,28801,0.544506
2,28802,0.524376


In [28]:
import lightgbm as lgb

# データをLightGBMのデータ構造に変換
lgb_train = lgb.Dataset(train_df.drop('target', axis=1), train_df['target'])
lgb_test = lgb.Dataset(test_df)

# LightGBMのパラメータを設定
params = {
    'objective': 'regression',  # 目的関数（回帰問題）
    'metric': 'rmse',          # 評価指標（RMSE）
    'boosting_type': 'gbdt',   # ブースティングタイプ
    'num_leaves': 31,          # 葉の数
    'learning_rate': 0.05,     # 学習率
    'feature_fraction': 0.9,   # 特徴量のサンプル率
    'bagging_fraction': 0.8,   # データのサンプル率
    'bagging_freq': 5,         # baggingの頻度
    'verbose': 0              # ログ出力レベル
}

# モデルの学習
gbm = lgb.train(params,
                lgb_train,
                num_boost_round=1000) # ブースト回数



[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [29]:
from sklearn.metrics import mean_squared_error

np.sqrt(mean_squared_error(target, gbm.predict(train_df.drop('target', axis=1))))

0.007207448490365836

In [30]:
# testデータで予測
lgbm_pred = gbm.predict(test_df, num_iteration=gbm.best_iteration)
lgbm_pred

array([0.40895949, 0.56643965, 0.66085791])

In [31]:
sub['prediction'] = lgbm_pred
# sub.to_csv('submission.csv')
sub

,ID,prediction
0,28800,0.408959
1,28801,0.566440
2,28802,0.660858


In [32]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# XGBoost用のデータセットを作成
dtrain = xgb.DMatrix(train_df.drop('target', axis=1), label=train_df['target'], enable_categorical=True)
dtest = xgb.DMatrix(test_df, enable_categorical = True)

# XGBoostのパラメータを設定
params = {
    'objective': 'reg:squarederror',  # 目的関数（回帰問題）
    'eval_metric': 'rmse',            # 評価指標（RMSE）
    'eta': 0.05,                      # 学習率
    'max_depth': 10,                   # 木の最大深度
    'subsample': 0.8,                 # データのサンプル率
    'colsample_bytree': 0.8,          # 特徴量のサンプル率
    'seed': 42,                        # 乱数シード
}

# モデルの学習
num_round = 1000  # ブースト回数
model_xgb = xgb.train(params, dtrain, num_round)

# RMSEの計算 (訓練データ)
xgb_pred_train = model_xgb.predict(dtrain)

rmse_xgboost_train = np.sqrt(mean_squared_error(target, xgb_pred_train))
print(f"XGBoost RMSE (Train): {rmse_xgboost_train}")

XGBoost RMSE (Train): 0.0006084528242543631


In [33]:
# テストデータで予測
xgb_pred = model_xgb.predict(dtest)
xgb_pred

array([0.69182295, 0.77272195, 0.6727893 ], dtype=float32)

In [34]:
# y_pred = catboost_pred*0.4+lgbm_pred*0.3+xgb_pred*0.3
y_pred = (catboost_pred+lgbm_pred+xgb_pred)/3

y_pred

array([0.51984219, 0.62788914, 0.61934116])

In [35]:
sub['prediction'] = y_pred
sub.to_csv('submission.csv', index = False)

In [36]:
sub

,ID,prediction
0,28800,0.519842
1,28801,0.627889
2,28802,0.619341
